# Step 2: Feature Engineering 


In [1]:
import sys
import os
import numpy as np
import pandas as pd

# Add Src to path
sys.path.insert(0, os.path.abspath(os.path.join('..', 'Src')))

from data.load_data import load_stock_prices, load_market_index, load_risk_free_rate
from data.clean_data import (
    clean_stock_prices,
    clean_market_index,
    clean_risk_free_rate,
    compute_log_returns,
    compute_market_log_return,
    align_datasets
)

RAW_DIR = os.path.join('..', 'Data', 'Raw')
INTERIM_DIR = os.path.join('..', 'Data', 'Interim')

---
## 1. Load Raw Data

In [2]:
stocks = load_stock_prices(RAW_DIR)
market = load_market_index(RAW_DIR)
rf = load_risk_free_rate(RAW_DIR)

print(f"Stock prices: {len(stocks):,} rows")
print(f"Market index: {len(market):,} rows")
print(f"Risk-free rate: {len(rf):,} rows")

Stock prices: 8,820 rows
Market index: 1,764 rows
Risk-free rate: 1,831 rows


---
## 2. Clean Data


In [3]:
# Clean each dataset
stocks_clean = clean_stock_prices(stocks)
market_clean = clean_market_index(market)
rf_clean = clean_risk_free_rate(rf)

  Risk-free rate: filled 78 NaN values


In [4]:
# Check for any remaining NaN
print("NaN counts after cleaning:")
print(f"  Stocks adj_close: {stocks_clean['adj_close'].isna().sum()}")
print(f"  Market adj_close: {market_clean['adj_close'].isna().sum()}")
print(f"  RF rate: {rf_clean['risk_free_rate'].isna().sum()}")

NaN counts after cleaning:
  Stocks adj_close: 0
  Market adj_close: 0
  RF rate: 0


---
## 3. Compute Log Returns



In [5]:
# Compute log returns
stocks_with_returns = compute_log_returns(stocks_clean, "adj_close", "ticker")
market_with_returns = compute_market_log_return(market_clean)

print("Log returns computed!")
stocks_with_returns[["date", "ticker", "adj_close", "log_return"]].head(10)

Log returns computed!


,date,ticker,adj_close,log_return
0,2019-01-02,AAPL,37.538826,NaN
1,2019-01-03,AAPL,33.799690,-0.104924
2,2019-01-04,AAPL,35.242554,0.041803
3,2019-01-07,AAPL,35.164120,-0.002228
4,2019-01-08,AAPL,35.834450,0.018883
5,2019-01-09,AAPL,36.442986,0.016839
6,2019-01-10,AAPL,36.559467,0.003191
7,2019-01-11,AAPL,36.200527,-0.009866
8,2019-01-14,AAPL,35.656174,-0.015151
9,2019-01-15,AAPL,36.385929,0.020260


In [6]:
# Sanity check: log returns should be small numbers (mostly -0.1 to 0.1)
print("Log return distribution:")
print(stocks_with_returns["log_return"].describe())

Log return distribution:
count    8815.000000
mean        0.000920
std         0.021243
min        -0.306391
25%        -0.009001
50%         0.001187
75%         0.011667
max         0.209308
Name: log_return, dtype: float64


In [7]:
# Check for extreme values (potential data issues)
extreme = stocks_with_returns[abs(stocks_with_returns["log_return"]) > 0.2]
print(f"\nExtreme returns (|r| > 20%): {len(extreme)} rows")
if len(extreme) > 0:
    print(extreme[["date", "ticker", "adj_close", "log_return"]].head())


Extreme returns (|r| > 20%): 3 rows
           date ticker   adj_close  log_return
6071 2022-02-03   META  236.110687   -0.306391
6255 2022-10-27   META   97.260605   -0.281794
6321 2023-02-02   META  187.460541    0.209308


---
## 4. Align Datasets

In [8]:
stocks_aligned, market_aligned, rf_aligned = align_datasets(
    stocks_with_returns, market_with_returns, rf_clean
)

print(f"\nAligned datasets:")
print(f"  Stocks: {len(stocks_aligned):,} rows")
print(f"  Market: {len(market_aligned):,} rows")

  Common dates between stocks and market: 1764



Aligned datasets:
  Stocks: 8,820 rows
  Market: 1,764 rows


---
## 5. Merge Market Return & RF Rate

In [9]:
# Add market return to stocks
market_returns = market_aligned[["date", "log_return"]].rename(
    columns={"log_return": "market_log_return"}
)
stocks_final = stocks_aligned.merge(market_returns, on="date", how="left")

# Add risk-free rate (convert to daily decimal)
# DTB3 is annualized %, so: daily = rate / 100 / 252
rf_aligned["rf_daily"] = rf_aligned["risk_free_rate"] / 100 / 252
rf_daily = rf_aligned[["date", "rf_daily"]]
stocks_final = stocks_final.merge(rf_daily, on="date", how="left")

# Fill any missing rf_daily
stocks_final["rf_daily"] = stocks_final["rf_daily"].ffill().bfill()

print("Merged market return and rf rate")

Merged market return and rf rate


In [10]:
# Drop first row of each ticker (NaN log_return)
stocks_final = stocks_final.dropna(subset=["log_return"])

# CRITICAL: Strict time ordering (no shuffling!)
stocks_final.sort_values(["ticker", "date"], inplace=True)
stocks_final.reset_index(drop=True, inplace=True)

print(f"Final shape: {stocks_final.shape}")

Final shape: (8815, 11)


---
## 6. Final Dataset

In [11]:
# Select final columns
final_cols = ["date", "ticker", "adj_close", "log_return", "market_log_return", "rf_daily"]
cleaned_data = stocks_final[final_cols]

cleaned_data.head(10)

,date,ticker,adj_close,log_return,market_log_return,rf_daily
0,2019-01-03,AAPL,33.799690,-0.104924,-0.025068,0.000094
1,2019-01-04,AAPL,35.242554,0.041803,0.033759,0.000094
2,2019-01-07,AAPL,35.164120,-0.002228,0.006986,0.000096
3,2019-01-08,AAPL,35.834450,0.018883,0.009649,0.000096
4,2019-01-09,AAPL,36.442986,0.016839,0.004090,0.000095
5,2019-01-10,AAPL,36.559467,0.003191,0.004508,0.000094
6,2019-01-11,AAPL,36.200527,-0.009866,-0.000146,0.000094
7,2019-01-14,AAPL,35.656174,-0.015151,-0.005271,0.000095
8,2019-01-15,AAPL,36.385929,0.020260,0.010665,0.000095
9,2019-01-16,AAPL,36.830452,0.012143,0.002220,0.000094


In [12]:
# Summary by ticker
print("Summary by ticker:")
cleaned_data.groupby("ticker").agg({
    "date": ["min", "max", "count"],
    "log_return": ["mean", "std"]
}).round(4)

Summary by ticker:


date                  log_return        
              min        max count       mean     std
ticker                                               
AAPL   2019-01-03 2026-01-07  1763     0.0011  0.0195
AMZN   2019-01-03 2026-01-07  1763     0.0006  0.0215
GOOG   2019-01-03 2026-01-07  1763     0.0010  0.0197
META   2019-01-03 2026-01-07  1763     0.0009  0.0266
MSFT   2019-01-03 2026-01-07  1763     0.0009  0.0179

---
## 7. Save Cleaned Data

In [13]:
# Ensure output dir exists
os.makedirs(INTERIM_DIR, exist_ok=True)

# Save
output_path = os.path.join(INTERIM_DIR, "cleaned_prices.csv")
cleaned_data.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

Saved: ..\Data\Interim\cleaned_prices.csv


In [14]:
print("\n" + "="*60)
print("STEP 2 COMPLETE")
print("Cleaning & Time-Series Consistency: PASSED")
print("="*60)


STEP 2 COMPLETE
Cleaning & Time-Series Consistency: PASSED
